In [30]:

import pandas as pd
import plotly.express as px

# Set global dimensions for all subsequent plotly express plots
px.defaults.width = 900
px.defaults.height = 500


In [31]:
x = pd.read_csv('NG_monthly9.csv')
x['Date'] = pd.to_datetime(x['Date'])
x['Production(bcm)'] = pd.to_numeric(x['Production(bcm)'], errors='coerce')
x['LNG_sendout(bcm)'] = pd.to_numeric(x['LNG_sendout(bcm)'], errors='coerce')
pipelist = ['BY', 'DZ', 'LY', 'MA', 'NO', 'RU-Baltic', 'TR', 'UA', 'DE', 'ES']
x['Piped'] = 0
for i in pipelist:
    x['Piped'] = x['Piped'] + x['bcm_' + i]
x['Net_supply'] = (
    x['LNG_sendout(bcm)'] 
    + x['Production(bcm)'] 
    + x['Piped'] 
    - x['CH_RS_net_exports'] 
    - x['bcm_ES']
    - x['Distribution_loss(bcm)']
)
x['Total_adj(bcm)'] = pd.to_numeric(x['Total_adj(bcm)'], errors='coerce')
x['Calc_storage_change'] = x['Net_supply'] - x['Total_adj(bcm)']
x['Act_storage_change'] = -x['Av_storage(bcm)'].diff(-1)
x['Diff_storage_changes'] = x['Act_storage_change'] - x['Calc_storage_change']

In [36]:
# Create a scatter plot with one variable on the x-axis and the other on the y-axis
fig = px.scatter(
    x, 
    x='Act_storage_change', 
    y='Calc_storage_change',
    title='Calculated storage changes vs Observed storage changes (bcm)',
    labels={
        'Act_storage_change': 'Observed storage change (bcm)', 
        'Calc_storage_change': 'Calculated storage change (bcm)'
    },
    trendline='ols'  # Optional: adds a trendline if you want to see the correlation
)

fig.show()

In [32]:

# 1. Ensure 'Date' is in datetime format so Plotly sorts and plots it correctly
x['Date'] = pd.to_datetime(x['Date'])

# 2. Reshape the dataframe from wide to long format for Plotly Express
df_melted = x.melt(
    id_vars=['Date'], 
    value_vars=['Calc_storage_change', 'Act_storage_change'],
    var_name='Metric', 
    value_name='Value'
)

# 3. Create the line plot
fig = px.line(
    df_melted, 
    x='Date', 
    y='Value', 
    color='Metric',
    title='Calculated storage changes vs Observed storage changes (bcm)',
    labels={'Value': 'bcm', 'Date': 'Date'}
)

# 4. Display the plot
fig.show()


In [37]:
# Create a line plot for diff_storage_changes over Date
fig = px.line(
    x, 
    x='Date', 
    y='Diff_storage_changes',
    title='Difference in Storage Changes over time',
    labels={
        'Date': 'Date', 
        'Diff_storage_changes': 'Observed minus calculated storage changes (bcm)'
    }
)

fig.show()

In [44]:
# 1. Ensure 'Date' is in datetime format
x['Date'] = pd.to_datetime(x['Date'])

# 2. Filter the dataframe to only include dates
Year = 2022
x_filtered = x[x['Date'] >= f'{Year}-01-01']

# 3. Create the scatter plot using the filtered data
fig = px.scatter(
    x_filtered, 
    x='Act_storage_change', 
    y='Calc_storage_change',
    title=f'Calculated storage changes vs Observed storage changes (bcm) from Jan {Year} onwards',
    labels={
        'Act_storage_change': 'Observed storage change (bcm)', 
        'Calc_storage_change': 'Calculated storage change (bcm)'
    },
    trendline='ols'  # Optional: adds a trendline if you want to see the correlation
)

fig.show()